# OpenICU-YAIB validation: MIMIC demo

This notebook follows the same workflow as `example/00_create_yaib_wide_and_compare_ricu.ipynb`:

1. export the RICU YAIB concepts and stay windows with the dataset-specific R script,
2. create the OpenICU YAIB-wide file for all available hours,
3. create the OpenICU YAIB-wide file for `0..168` hours,
4. normalize comparison dtypes and compare the 168-hour OpenICU file with RICU.

Run the R export from the repository root first:

```bash
RICU_OUT_DIR="$HOME/output/openicu_yaib/mimic_demo" \
Rscript scripts/datasets/export_ricu_mimic_demo.R
```

The R script produces both:

- `ricu_dynamic_vars_mimic_demo.parquet`
- `ricu_stay_windows_mimic_demo.parquet`

The notebook automatically maps `subject_id` and `time` through the dataset-specific ICU-stay table; no transformation code must be added by the user.


In [ ]:
from pathlib import Path

from openicu_yaib import build_and_write_yaib_wide_for_dataset

# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------
DATASET = "mimic_demo"
RICU_SRC = "mimic_demo"
OPENICU_CONCEPT_VERSION = "1.0.0"
MAX_HOURS = 7 * 24

# Keep every dataset in its own output directory.
OUTPUT_ROOT = Path.home() / "output" / "openicu_yaib" / DATASET

# Optional path overrides.
CONCEPT_ROOT = None
ICUSTAYS_CSV = None
RICU_CONCEPT_DICT = None

# The workflow resolves the dataset-specific ICU-stay table automatically
# from OPENICU_YAIB_<DATASET>_STAYS, OPENICU_YAIB_DATA_ROOT, or RICU_DATA_PATH.
INCLUDE_GRID = True

# Match the original notebook: every required YAIB concept must be available.
MISSING_CONCEPTS = "fail"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATASET}")
print(f"Output:  {OUTPUT_ROOT}")


In [ ]:
# ---------------------------------------------------------------------------
# 1) Export all available ICU hours.
# ---------------------------------------------------------------------------
all_hours = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=None,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    include_grid=INCLUDE_GRID,
    missing_concepts=MISSING_CONCEPTS,
)

print(f"Wrote all-hours OpenICU YAIB-wide parquet to: {all_hours.output_path}")
all_hours.summary


In [ ]:
# ---------------------------------------------------------------------------
# 2) Export the first seven days for R/RICU validation.
#
# The maximum is inclusive: time = 0, 1, ..., 168.
# ---------------------------------------------------------------------------
one_week = build_and_write_yaib_wide_for_dataset(
    dataset=DATASET,
    version=OPENICU_CONCEPT_VERSION,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    concept_root=CONCEPT_ROOT,
    icustays_csv=ICUSTAYS_CSV,
    ricu_concept_dict=RICU_CONCEPT_DICT,
    include_grid=INCLUDE_GRID,
    missing_concepts=MISSING_CONCEPTS,
)

print(f"Wrote one-week OpenICU YAIB-wide parquet to: {one_week.output_path}")
one_week.summary


## Compare with R/`ricu`


In [ ]:
from openicu_yaib import (
    compare_openicu_wide_to_ricu_for_dataset,
    display_comparison_overview,
)

# ---------------------------------------------------------------------------
# 3) Compare the 168-hour OpenICU table with the R/RICU reference.
#
# The comparison helper normalizes stay_id/time to Int64 and all shared
# dynamic variables to the same floating-point dtype before comparison.
# It also limits the RICU stay windows and dynamic table to time <= 168.
# ---------------------------------------------------------------------------
ricu_dynamic = OUTPUT_ROOT / f"ricu_dynamic_vars_{RICU_SRC}.parquet"
ricu_windows = OUTPUT_ROOT / f"ricu_stay_windows_{RICU_SRC}.parquet"

if not ricu_dynamic.is_file() or not ricu_windows.is_file():
    raise FileNotFoundError(
        "RICU reference files are missing. Run the dataset-specific R script first:\n"
        f"  {ricu_dynamic}\n"
        f"  {ricu_windows}"
    )

comparison = compare_openicu_wide_to_ricu_for_dataset(
    dataset=DATASET,
    max_hours=MAX_HOURS,
    output_root=OUTPUT_ROOT,
    openicu_wide_path=one_week.output_path,
    ricu_dynamic_path=ricu_dynamic,
    ricu_stay_windows_path=ricu_windows,
)

print(f"Wrote comparison reports to: {comparison.reports_dir}")
display_comparison_overview(comparison)
